In [1]:
pip install --upgrade transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 107.1 MB/s eta 0:00:0000:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.4
    Uninstalling transformers-4.52.4:
      Successfully uninstalled transformers-4.52.4
^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install datasets evaluate rouge-score sacrebleu bertscore


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.5 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement bertscore (from versions: none)
ERROR: No matching distribution found for bertscore
Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

2025-07-10 17:19:38.730961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752167978.912465      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752167978.962449      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [7]:
raw_train = load_dataset("euclaise/writingprompts", split="train")

raw_train = raw_train.select(range(20000))

split_dataset = raw_train.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

In [8]:
# Check the size of the datasets
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")


Training dataset size: 16000
Validation dataset size: 4000


In [9]:
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [10]:
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)
trainer.train()
trainer.save_model("./content_creation30")
tokenizer.save_pretrained("./content_creation30")
!zip -r content_creation30.zip ./content_creation30


/tmp/ipykernel_36/2658365815.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.229300,3.118629
2,3.106300,3.097640
3,3.051200,3.091729


  adding: content_creation30/ (stored 0%)
  adding: content_creation30/tokenizer.json (deflated 82%)
  adding: content_creation30/training_args.bin (deflated 52%)
  adding: content_creation30/config.json (deflated 52%)
  adding: content_creation30/generation_config.json (deflated 24%)
  adding: content_creation30/tokenizer_config.json (deflated 54%)
  adding: content_creation30/merges.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 53%)
  adding: content_creation30/special_tokens_map.json (deflated 60%)
  adding: content_creation30/vocab.json (deflated 59%)
  adding: content_creation30/model.safetensors (deflated 7%)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



Usage:   
  /usr/bin/python3 -m pip <command> [options]

no such option: --upgrade
Note: you may need to restart the kernel to use updated packages.


In [27]:
model = AutoModelForCausalLM.from_pretrained("./content_creation30")
tokenizer = AutoTokenizer.from_pretrained("./content_creation30")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"][:500]
true_stories = val_dataset["story"][:500]

model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A robot wakes up on a deserted planet.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=512,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

BLEU: 0.38
ROUGE-L: 0.09
Perplexity: 24.6859


ImportError: To be able to use evaluate-metric/bertscore, you need to install the following dependencies['bert_score'] using 'pip install bert_score' for instance'

In [30]:
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A robot wakes up on a deserted planet.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=512,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BERTScore F1: 0.7936928498744965
Prompt: A robot wakes up on a deserted planet.
 `` Hello Mr. Robot, my name is Alex and I'm here to tell you how this was supposed to go.'' The voice sounded like a doctor speaking into a speaker. I was n't sure what was going on. I was just curious as to why Mr. Robot woke up in a desert desert. 
 
 `` I was just wondering if anyone is out there to help me out?'' 
 
 `` Yes sir.'' 
 
 `` Do you really think that's what happens when you go to work?'' 
 
 `` You think I'm crazy?'' 
 
 `` No sir.'' 
 
 `` Well sir, do you know how many people you've lost?'' 
 
 `` Thirty eight. You can only do this once.'' 
 
 `` Do you really think that a robot can survive a war that lasts days on end?'' 
 
 `` No sir.'' 
 
 `` I did n't know that.'' 
 
 `` Oh well I suppose not.'' 
 
 `` So what if I tell you you've been here since noon and I wake up on a deserted planet with no one to help me?'' 
 
 `` No sir.'' 
 
 `` You are n't being ridiculous Mr. Robot.'' 
 
 `` Y

In [31]:
model_name = "distilgpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer2 = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)
trainer2.train()
trainer2.save_model("./content_creation30-2")
tokenizer.save_pretrained("./content_creation30-2")
!zip -r content_creation30-2.zip ./content_creation30-2


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

/tmp/ipykernel_36/1795860749.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer2 = Trainer(


Epoch,Training Loss,Validation Loss
1,3.480800,3.350915
2,3.363900,3.321472
3,3.316000,3.314161


  adding: content_creation30-2/ (stored 0%)
  adding: content_creation30-2/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 82%)
  adding: content_creation30-2/training_args.bin (deflated 52%)
  adding: content_creation30-2/config.json (deflated 52%)
  adding: content_creation30-2/generation_config.json (deflated 24%)
  adding: content_creation30-2/tokenizer_config.json (deflated 54%)
  adding: content_creation30-2/merges.txt (deflated 53%)
  adding: content_creation30-2/special_tokens_map.json (deflated 60%)
  adding: content_creation30-2/vocab.json (deflated 59%)
  adding: content_creation30-2/model.safetensors (deflated 7%)


In [32]:
model = AutoModelForCausalLM.from_pretrained("content_creation30-2")
tokenizer = AutoTokenizer.from_pretrained("content_creation30-2")
bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]
perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))



device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A robot wakes up on a deserted planet.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=512,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

BLEU: 0.38
ROUGE-L: 0.09
Perplexity: 30.5704


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


BERTScore F1: 0.7936928498744965
Prompt: A robot wakes up on a deserted planet.
 `` Hi,'' the human voice said, `` What is it?'' 
 
 `` It's a robot, you got ta get the rest of you.'' 
 
 `` Yes.'' 
 
 `` We are going to kill it.'' 
 
 `` It's a robot!'' 
 
 `` That is the robot. We are going to kill it.'' 
 
 `` It is a robot. We are going to kill it. It is a robot, you got ta get the rest of you.'' 
 
 `` The robot is still alive. It is a robot, you got ta get the rest of you.'' 
 
 `` But not you, the robot is still alive.'' 
 
 `` You got ta get the rest of you.'' 
 
 `` The robot is still alive.'' 
 
 `` I'm sorry, this robot was supposed to be dead, I did n't get any time. I just did n't have time to think of how many more things you need to know.'' 
 
 `` So I'm going to kill the robot.'' 
 
 `` But you do n't have time to think about how many more things you need to know.'' 
 
 `` Okay, let's just start over, we are going to kill it.'' 
 
 `` That is the robot.'' 
 
 `` Well, I

In [ ]:
model_name = "EleutherAI/pythia-160m"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=512)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer3 = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)
trainer3.train()
trainer3.save_model("./content_creation30-3")
tokenizer.save_pretrained("./content_creation30-3")
!zip -r content_creation30-3.zip ./content_creation30-3

In [ ]:
model = AutoModelForCausalLM.from_pretrained("content_creation30-3")
tokenizer = AutoTokenizer.from_pretrained("content_creation30-3")
bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]
perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))



device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A robot wakes up on a deserted planet.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=512,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


In [21]:
model_name = "/kaggle/working/content_creation"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

In [22]:
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [24]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


In [28]:
trainer.train()
trainer.save_model("./content_creation2")
tokenizer.save_pretrained("./content_creation2")
!zip -r content_creation2.zip ./content_creation2


Epoch,Training Loss,Validation Loss
1,3.065800,3.125058


  adding: content_creation2/ (stored 0%)
  adding: content_creation2/special_tokens_map.json (deflated 74%)
  adding: content_creation2/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)
  adding: content_creation2/generation_config.json (deflated 24%)
  adding: content_creation2/vocab.json (deflated 59%)
  adding: content_creation2/tokenizer.json (deflated 82%)
  adding: content_creation2/config.json (deflated 52%)
  adding: content_creation2/tokenizer_config.json (deflated 57%)
  adding: content_creation2/training_args.bin (deflated 52%)
  adding: content_creation2/merges.txt (deflated 53%)


In [ ]:
# Replace with your own tokenized test set
metrics = trainer.evaluate(eval_dataset=tokenized_test)
print(metrics)


In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("./content_creation")
tokenizer = AutoTokenizer.from_pretrained("./content_creation")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"]
true_stories = val_dataset["story"]
model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    
import evaluate
import math
import torch

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

BLEU: 0.42
ROUGE-L: 0.09
Perplexity: 24.1553


In [38]:
!pip install evaluate
!pip install bert_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:01m0:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.6 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 44.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing inst

In [39]:
import evaluate

bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.7933246456384658


In [52]:
# Initialize with proper device handling
tokenizer = AutoTokenizer.from_pretrained("./content_creation")
model = AutoModelForCausalLM.from_pretrained("./content_creation")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A man wakes up on a deserted island.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=128,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: A man wakes up on a deserted island.
 This guy woke up like everyone else on that island. He was in the back of his ship, and all he could think was how he could make it to the other end of the ship. He did n't want to be there; he needed to get home. He needed to find a way to leave. 
 
 
 There are some places where you can sleep on empty. The back door is open, but the only one he has is on the other side of it. There is no way to get home, so he made his way to the


In [53]:

model = AutoModelForCausalLM.from_pretrained("./content_creation2")
tokenizer = AutoTokenizer.from_pretrained("./content_creation2")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"]
true_stories = val_dataset["story"]
model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

BLEU: 0.42
ROUGE-L: 0.09
Perplexity: 24.384


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.7931430215239524


In [2]:
pip install datasets evaluate rouge-score sacrebleu


Note: you may need to restart the kernel to use updated packages.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch
raw_train = load_dataset("euclaise/writingprompts", split="train")

raw_train = raw_train.select(range(10000))

split_dataset = raw_train.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
# Check the size of the datasets
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
model_name = "distilgpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


2025-06-25 18:21:45.985268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750875706.187959      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750875706.245852      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


README.md:   0%|          | 0.00/837 [00:00<?, ?B/s]

(…)-00000-of-00002-105e07cb0d199464.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

(…)-00001-of-00002-4fdb982c11056472.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

(…)-00000-of-00001-16503b0c26ed00c6.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_35/528849018.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [5]:
trainer.train()
trainer.save_model("./content_creation-small")
tokenizer.save_pretrained("./content_creation-small")
!zip -r content_creation-small.zip ./content_creation-small

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.508800,3.384867
2,3.384500,3.359564
3,3.335200,3.354442


  adding: content_creation-small/ (stored 0%)
  adding: content_creation-small/tokenizer_config.json (deflated 54%)
  adding: content_creation-small/training_args.bin (deflated 52%)
  adding: content_creation-small/generation_config.json (deflated 24%)
  adding: content_creation-small/vocab.json (deflated 59%)
  adding: content_creation-small/config.json (deflated 52%)
  adding: content_creation-small/special_tokens_map.json (deflated 60%)
  adding: content_creation-small/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 82%)
  adding: content_creation-small/merges.txt (deflated 53%)
  adding: content_creation-small/model.safetensors (deflated 7%)


In [10]:
import evaluate
import math
import torch

model = AutoModelForCausalLM.from_pretrained("./content_creation-small")
tokenizer = AutoTokenizer.from_pretrained("./content_creation-small")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"]
true_stories = val_dataset["story"]
model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

BLEU: 0.40
ROUGE-L: 0.09
Perplexity: 30.2741


ImportError: To be able to use evaluate-metric/bertscore, you need to install the following dependencies['bert_score'] using 'pip install bert_score' for instance'

In [14]:
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.7894456019997597


In [13]:
pip install bert_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing ins

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch
raw_train = load_dataset("euclaise/writingprompts", split="train")

raw_train = raw_train.select(range(10000))

split_dataset = raw_train.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
# Check the size of the datasets
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
model_name = "/kaggle/working/content_creation-small"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer1 = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


Training dataset size: 9000
Validation dataset size: 1000


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_35/2224379102.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer1 = Trainer(


In [7]:
trainer1.train()
trainer1.save_model("./content_creation-small2")
tokenizer.save_pretrained("./content_creation-small2")
!zip -r content_creation-small2.zip ./content_creation-small2

Epoch,Training Loss,Validation Loss
1,3.301700,3.356187


  adding: content_creation-small2/ (stored 0%)
  adding: content_creation-small2/tokenizer_config.json (deflated 57%)
  adding: content_creation-small2/training_args.bin (deflated 52%)
  adding: content_creation-small2/generation_config.json (deflated 24%)
  adding: content_creation-small2/vocab.json (deflated 59%)
  adding: content_creation-small2/config.json (deflated 52%)
  adding: content_creation-small2/special_tokens_map.json (deflated 74%)
  adding: content_creation-small2/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 82%)
  adding: content_creation-small2/merges.txt (deflated 53%)
  adding: content_creation-small2/model.safetensors (deflated 7%)


In [20]:
# Initialize with proper device handling
tokenizer = AutoTokenizer.from_pretrained("./content_creation-small2")
model = AutoModelForCausalLM.from_pretrained("./content_creation-small2")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A robot wakes up on a deserted island.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=128,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: A robot wakes up on a deserted island. But why? Why? Why do n't they have to resort to violence? The robots are humans. They have been trained to be deadly, they know that when they die they will not be given life. And it will not be easy. There are many people in this world, but they have to survive. And so they must survive. They must survive. 
 
 I do n't know why, but I have n't been able to do it. I've found many ways to survive. I've found a place in a forest, a land mine,


In [1]:
pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing in

In [3]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import evaluate
import math
import torch

model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"]
true_stories = val_dataset["story"]
model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


BLEU: 0.30
ROUGE-L: 0.09
Perplexity: 33.118


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.763794000685215


In [11]:
model2 = AutoModelForCausalLM.from_pretrained("distilgpt2")
tokenizer2 = AutoTokenizer.from_pretrained("distilgpt2")
bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]
perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

BLEU: 0.30
ROUGE-L: 0.09
Perplexity: 33.118


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.763794000685215


In [27]:
model_name = "EleutherAI/pythia-160m"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=512)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer1 = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_35/4068303246.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer1 = Trainer(


In [28]:
trainer1.train()
trainer1.save_model("./content_creation-big")
tokenizer.save_pretrained("./content_creation-big")
!zip -r content_creation-big.zip ./content_creation-big

Epoch,Training Loss,Validation Loss
1,3.650200,3.566173
2,3.217200,3.464305
3,2.853000,3.464046


  adding: content_creation-big/ (stored 0%)
  adding: content_creation-big/config.json (deflated 51%)
  adding: content_creation-big/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 81%)
  adding: content_creation-big/model.safetensors (deflated 9%)
  adding: content_creation-big/training_args.bin (deflated 52%)
  adding: content_creation-big/tokenizer_config.json (deflated 92%)
  adding: content_creation-big/special_tokens_map.json (deflated 75%)
  adding: content_creation-big/generation_config.json (deflated 23%)


In [29]:
import evaluate
import math
import torch

model = AutoModelForCausalLM.from_pretrained("content_creation-big")
tokenizer = AutoTokenizer.from_pretrained("content_creation-big")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"]
true_stories = val_dataset["story"]
model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 

BLEU: 0.49
ROUGE-L: 0.09
Perplexity: 34.3516


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.7898055878281594


In [11]:
model_name = "/kaggle/input/content_creation-big/other/default/1/content_creation-big"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=512)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer1 = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)
trainer1.train()
trainer1.save_model("./content_creation-big2")
tokenizer.save_pretrained("./content_creation-big2")
!zip -r content_creation-big.zip ./content_creation-big

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_35/3510173449.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer1 = Trainer(


Epoch,Training Loss,Validation Loss
1,3.103000,3.651479
2,2.719100,3.676055
3,2.297700,3.861524


	zip warning: name not matched: ./content_creation-big

zip error: Nothing to do! (try: zip -r content_creation-big.zip . -i ./content_creation-big)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
!pip install evaluate
import evaluate

  Using cached evaluate-0.4.4-py3-none-any.whl.metadata (9.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.9.0.13 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_sys

In [68]:
!pip install rouge_score bert_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [66]:
import random
import evaluate
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score,recall_score
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline, AutoModelForSequenceClassification
from tqdm import tqdm
import pandas as pd
from datasets import Dataset, load_dataset
from openai import OpenAI


In [79]:

# Load Evaluation Metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
perplexity_metric = evaluate.load("perplexity")
bertscore = evaluate.load("bertscore")

# --- Generalized Sampling Function ---
def sample_data(dataset, n=25):
    dataset_list = list(dataset)
    return random.sample(dataset_list, min(n, len(dataset_list)))

# --- Task-Specific Inference Helpers ---
def slm_inference(model_pipeline, inputs, batch_size=8, task_type="classification"):
    results = []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i+batch_size]
        if task_type == "classification":
            batch_results = model_pipeline(batch, truncation=True, padding=True)
            results.extend([int(res["label"].replace("LABEL_", "")) for res in batch_results])
        elif task_type == "generation":
            generated = model_pipeline(batch, max_length=1000)
            results.extend([g[0]["generated_text"] for g in generated])
    return results

def llm_inference(api_call_func, inputs, batch_size=8):
    results = []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i+batch_size]
        batch_results = api_call_func(batch)
        if not batch_results:
            print(f"[WARNING] No results for batch {i//batch_size}")
        results.extend(batch_results)
    return results


# --- Generalized Metric Calculation ---
def calculate_metrics(predictions, references, task_type):
    results = {}
    if task_type == "generation":
        # Existing metrics
        results['BLEU'] = bleu.compute(predictions=predictions, references=[[ref] for ref in references])["bleu"]
        results['ROUGE-L'] = rouge.compute(predictions=predictions, references=references)["rougeL"]
        results['Perplexity'] = perplexity_metric.compute(predictions=predictions, model_id="gpt2")["perplexities"][0]

        # New: BERTScore
        bertscore_results = bertscore.compute(predictions=predictions, references=references, lang="en")
        results['BERTScore_P'] = sum(bertscore_results['precision']) / len(bertscore_results['precision'])
        results['BERTScore_R'] = sum(bertscore_results['recall']) / len(bertscore_results['recall'])
        results['BERTScore_F1'] = sum(bertscore_results['f1']) / len(bertscore_results['f1'])
    elif task_type == "classification":
        results['Accuracy'] = accuracy_score(references, predictions)
        results['F1-Score'] = f1_score(references, predictions, average='weighted',zero_division=0)
        results['Precision'] = precision_score(references, predictions, average='weighted',zero_division=0)
        results['Recall'] = recall_score(references, predictions, average='weighted',zero_division=0)
    return results


# --- Model Loading Helpers ---
def load_slm(model_path, task_type):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if task_type == "classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        return pipeline("text-classification", model=model, tokenizer=tokenizer)
    elif task_type == "generation":
        return pipeline("text-generation", model=model_path, tokenizer=tokenizer)

# --- Generalized Evaluation Function ---
def evaluate_task(models, dataset, task_config):
    sampled_data = sample_data(dataset)
    inputs = [item[task_config["input_key"]] for item in sampled_data]
    references_all = [item[task_config["ref_key"]] for item in sampled_data]

    results = []

    # Load tokenizer for truncation
    tokenizer = AutoTokenizer.from_pretrained("gpt2")

    def truncate(text, max_tokens=300):
        if not isinstance(text, str) or text.strip() == "":
            return ""
        tokens = tokenizer.encode(text, truncation=True, max_length=max_tokens)
        return tokenizer.decode(tokens, skip_special_tokens=True) if tokens else ""

    for name, model in models.items():
        print(f"\n[EVALUATING] Model: {name}")
        
        if task_config["model_type"] == "slm":
            preds = slm_inference(model, inputs, task_type=task_config["task_type"])
        else:
            preds = llm_inference(model, inputs)

        # Truncate all
        preds = [truncate(p) for p in preds]
        references = [truncate(r) for r in references_all]

        # Filter out empty or invalid pairs
        cleaned_pairs = [
            (p, r) for p, r in zip(preds, references)
            if isinstance(p, str) and isinstance(r, str) and p.strip() and r.strip()
        ]

        if not cleaned_pairs:
            print(f"[WARNING] Skipping model {name} — all predictions were empty or invalid.")
            continue

        if len(cleaned_pairs) < 10:
            print(f"[WARNING] Skipping model {name} — too few valid predictions ({len(cleaned_pairs)}).")
            continue

        preds, references = zip(*cleaned_pairs)  # unzip

        # Compute metrics
        metrics = calculate_metrics(preds, references, task_config["task_type"])
        results.append({"Model": name, **metrics})

    return pd.DataFrame(results)

# === Task Configurations ===
TASK_CONFIGS = {
    "sentiment": {
        "input_key": "text",
        "ref_key": "label",
        "task_type": "classification",
        "slm_paths": {
            "pre": "roberta-base",
            "post": "/home/deepan/AnanditaUma/sentimentAnalysis/sentimentanalysis"
        }
    },
    "translation": {
        "input_key": "english_sentence",
        "ref_key": "hindi_sentence",
        "task_type": "generation",
        "slm_paths": {
            "pre": "mbart-large-50",
            "post": "/home/deepan/AnanditaUma/translation/model_with_lora4/"
        }
    },
    "content": {
        "input_key": "prompt",
        "ref_key": "story",
        "task_type": "generation",
        "slm_paths": {
            "pre": "gpt2",
            "post": "/home/deepan/AnanditaUma/contentcreation/gpt2-writingprompts2/"
        }
    }
}

# --- Task Evaluation Functions ---
def evaluate_slm(task_name, test_set):
    config = TASK_CONFIGS[task_name]
    models = {
        "SLM Pre": load_slm(config["slm_paths"]["pre"], config["task_type"]),
        "SLM Post": load_slm(config["slm_paths"]["post"], config["task_type"])
    }
    return evaluate_task(models, test_set, {**config, "model_type": "slm"})

def evaluate_llm(task_name, test_set, api_functions):
    config = TASK_CONFIGS[task_name]
    models = {name: fn for name, fn in api_functions.items()}
    return evaluate_task(models, test_set, {**config, "model_type": "llm"})


In [70]:
import re
import time
import logging
from openai import OpenAI
import random


# Define system prompt for LLM to rate sentiment on a 0–4 scale
system_prompt = (
    "You are a sentiment classifier. Read the review and respond with ONLY a single integer (0–4) based on the following:\n"
    "0 = Very Negative, 1 = Negative, 2 = Neutral, 3 = Positive, 4 = Very Positive.\n"
    "Reply with only the integer."
)

# Init OpenAI client via OpenRouter
ds_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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" 
)

# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Robust scoring function with fallback
def safe_deepseek_sentiment(batch):
    responses = []
    for text in batch:
        try:
            response = ds_client.chat.completions.create(
                model="deepseek/deepseek-r1-0528:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            logger.info(f"Raw response: {content}")
            print(f"INPUT: {text}\nRESPONSE: {content}\n")


            match = re.search(r"\b[0-4]\b", content)
            if match:
                label = int(match.group(0))
                responses.append(label)
            else:
                raise ValueError(f"No valid sentiment label found in response: '{content}'")

        except Exception as e:
            logger.error(f"Error processing input {len(responses)}: {text}\nException: {e}")
            responses.append(2)  # Default to Neutral
            time.sleep(1)
    return responses

# Mapping model name to function
sentiment_llms = {
    "deepseek/deepseek-r1-0528:free": safe_deepseek_sentiment
}


In [42]:
full_test_set_sa = load_dataset("yelp_review_full", split="train")
test_set_sa = full_test_set_sa.shuffle(seed=42).select(range(25))
sentiment_results_llm = evaluate_llm("sentiment", test_set_sa, sentiment_llms)
print(sentiment_results_llm)

INPUT: We recently adopted our dog from my mom-in-law who lives in Casa Grande and I wanted to get Bruno in to see a vet in Tempe. I looked up a few offices and saw this location had an awesome rating. I was pleased to see someone I respect, Shayna K. from Scottsdale Doggie Suites, providing a 5 star rating and that was it. \n\nI immediately decided to make a new patient appointment for my furry baby. The office is clean, the staff is nice, and they made Bruno feel right at home. \n\nDr. Berthiaume came in and was very informative. I think he could tell I was a nervous \"new mom\" and he helped me stay at ease. He suggested one medication for Bruno which is for heartworm prevention but didn't pressure me into making a purchase right then and there. He gave me a pamplet on it and told me to do some research on my own first. (AMEN!)\n\nWhen I told them about my concern about him scratching his ears a lot, they took swabs but didn't run them. Dr. B. said if he saw anything \"bad\" he woul

In [ ]:
full_test_set_sa = load_dataset("yelp_review_full", split="train")
test_set_sa = full_test_set_sa.shuffle(seed=42).select(range(25))
sentiment_results_llm = evaluate_llm("sentiment", test_set_sa, sentiment_llms)
print(sentiment_results_llm)

In [80]:
from openai import OpenAI
import openai
# Prompt for the model
system_prompt = (
    "You are a short story genrator. read the prompt given and based on that generate a short story. It should not be longer than 300 words."
)

# Init OpenAI client with OpenRouter
ds_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")


# Define the robust sentiment function
def safe_deepseek_content(batch):
    responses = []
    for i, text in enumerate(batch):
        try:
        
            response = ds_client.chat.completions.create(
                model="deepseek/deepseek-r1-0528:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            print(f"[DEBUG] Prompt {i}: {text[:30]}...\nResponse: {content[:50]}...\n")
            responses.append(content)

        except Exception as e:
            print(f"[ERROR] Failed to process input {i}: {text[:50]}\nException: {e}")
            responses.append("")

    return responses
# Hook it up to your eval
content_llms = {
    "deepseek/deepseek-r1-0528:free": safe_deepseek_content
}
test_set_cc = val_dataset.shuffle(seed=42).select(range(20))
# Evaluate LLM predictions
content_results_llm = evaluate_llm("content", test_set_cc, content_llms)
print(content_results_llm)


[EVALUATING] Model: deepseek/deepseek-r1-0528:free
[DEBUG] Prompt 0: [ WP ] The U.S. gets hit by an...
Response: The tremor that hit Philadelphia wasn't the proble...

[DEBUG] Prompt 1: [ WP ] You have the power to s...
Response: The lukewarm dregs of my fifth coffee sloshed over...

[DEBUG] Prompt 2: [ WP ] Your bow does n't shoot...
Response: The weight of Great-Uncle Borin’s bow felt awkward...

[DEBUG] Prompt 3: [ WP ] Four years ago , your d...
Response: The familiar weight against my knee was the first ...

[DEBUG] Prompt 4: [ WP ] Try to get away with mu...
Response: The rain tracked down the interrogation room windo...

[DEBUG] Prompt 5: [ WP ] In the 1700s , an etern...
Response: The moon bled silver over the Roaring Forties as t...

[DEBUG] Prompt 6: [ WP ] Humanity is the only ``...
Response: Archivist K’lar of the Kelpian Collective adjusted...

[DEBUG] Prompt 7: [ WP ] Write about the most un...
Response: ## Form D-312b vs. The Limerick Lady

Mortimer Ble...

[DEBUG] Prom

  0%|          | 0/2 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


                            Model     BLEU   ROUGE-L  Perplexity  BERTScore_P  \
0  deepseek/deepseek-r1-0528:free  0.01182  0.107593   76.016197     0.802815   

   BERTScore_R  BERTScore_F1  
0     0.807836      0.805278  


In [ ]:
from openai import OpenAI
import openai
# Prompt for the model
system_prompt = (
    "You are a short story genrator. read the prompt given and based on that generate a short story. It should not be longer than 300 words."
)

# Init OpenAI client with OpenRouter
ds_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")


# Define the robust sentiment function
def safe_deepseek_content(batch):
    responses = []
    for i, text in enumerate(batch):
        try:
        
            response = ds_client.chat.completions.create(
                model="deepseek/deepseek-r1-0528:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            print(f"[DEBUG] Prompt {i}: {text[:30]}...\nResponse: {content[:50]}...\n")
            responses.append(content)

        except Exception as e:
            print(f"[ERROR] Failed to process input {i}: {text[:50]}\nException: {e}")
            responses.append("")

    return responses
# Hook it up to your eval
content_llms = {
    "deepseek/deepseek-r1-0528:free": safe_deepseek_content
}
test_set_cc = val_dataset.shuffle(seed=42).select(range(20))
# Evaluate LLM predictions
content_results_llm = evaluate_llm("content", test_set_cc, content_llms)
print(content_results_llm)


[EVALUATING] Model: deepseek/deepseek-r1-0528:free
[DEBUG] Prompt 0: [ WP ] You understand why the ...
Response: The chrysanthemums on the kitchen table were wilti...

[DEBUG] Prompt 1: [ WP ] Your i-pod is sentient ...
Response: The old iPod felt heavier in Liam's pocket that Ju...

[DEBUG] Prompt 2: [ WP ] You have the power to s...
Response: The air went still, the hiss of espresso machines ...

[DEBUG] Prompt 3: [ WP ] A knight saves a prince...
Response: Sir Aric of Elderglen spurred his charger, Ember, ...

[DEBUG] Prompt 4: [ WP ] In the 1700s , an etern...
Response: The moonlight bled silver onto the docks of the na...

[DEBUG] Prompt 5: [ WP ] Four years ago , your d...
Response: The doorbell chimed, a sound as jarring as shatter...

[DEBUG] Prompt 6: [ WP ] You woke up with no ind...
Response: The cold metal of the window frame bites into my p...

[DEBUG] Prompt 7: [ WP ] Iron is the only elemen...
Response: The archive cavern was vast, crystalline, the air ...

[DEBUG] Prom

In [78]:

# Prompt for the model
system_prompt = (
    "You are a short story genrator. read the prompt given and based on that generate a short story. It should not be longer than 300 words."
)

# Init OpenAI client with OpenRouter
ds_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Define the robust sentiment function
def safe_qwen_content(batch):
    responses = []
    for i, text in enumerate(batch):
        try:
        
            response = ds_client.chat.completions.create(
                model="qwen/qwen3-235b-a22b:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            print(f"[DEBUG] Prompt {i}: {text[:30]}...\nResponse: {content[:50]}...\n")
            responses.append(content)

        except Exception as e:
            print(f"[ERROR] Failed to process input {i}: {text[:50]}\nException: {e}")
            responses.append("")
    return responses

# Hook it up to your eval
content_llms3= {
    "qwen/qwen3-235b-a22b:free": safe_qwen_content
}
content_results_llm = evaluate_llm("content", test_set_cc, content_llms3)
print(content_results_llm)

[DEBUG] Prompt 0: [ WP ] He stood there , lookin...
Response: **The Grave on November First**  

He stood there,...

[DEBUG] Prompt 1: [ WP ] In the 1700s , an etern...
Response: **Title: Hook’s Redemption**  

In the squalid por...

[DEBUG] Prompt 2: [ WP ] You woke up with no ind...
Response: You wake gasping, palms pressed to cold steel, a m...

[DEBUG] Prompt 3: [ WP ] Iron is the only elemen...
Response: **The Iron Chain**  

The ancient texts lied. Eart...

[DEBUG] Prompt 4: [ WP ] You finally have an ext...
Response: **Title: The Diplomacy of Desperation**  

I’d tri...

[DEBUG] Prompt 5: [ WP ] Try to get away with mu...
Response: **Title: "Chain of Custody"**  

Detective Mara Ca...

[DEBUG] Prompt 6: [ WP ] You have the power to s...
Response: **Title: Unpaused**  

I glanced at my watch. 10:0...

[DEBUG] Prompt 7: [ WP ] The apocalypse happened...
Response: **Title: "The Last Orbit"**  

After the sun died ...

[DEBUG] Prompt 0: [ WP ] A knight saves a prince...
Response: **

  0%|          | 0/2 [00:00<?, ?it/s]

                       Model    BLEU   ROUGE-L  Perplexity  BERTScore_P  \
0  qwen/qwen3-235b-a22b:free  0.0123  0.121797   58.858803     0.808371   

   BERTScore_R  BERTScore_F1  
0     0.815636      0.811951  


In [62]:

# Prompt for the model
system_prompt = (
    "You are a short story genrator. read the prompt given and based on that generate a short story. It should not be longer than 300 words."
)

# Init OpenAI client with OpenRouter
ds_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Define the robust sentiment function
def safe_mistral_content(batch):
    responses = []
    for i, text in enumerate(batch):
        try:
            response = ds_client.chat.completions.create(
                model="mistralai/mistral-small-3.1-24b-instruct-2503:free",
                messages=[
                    {"role": "system", "content": "You're a creative writer. Complete the prompt:"},
                    {"role": "user", "content": text}
                ]
            )
            content = response.choices[0].message.content.strip()
            print(f"[DEBUG] Prompt {i}: {text[:30]}...\nResponse: {content[:50]}...\n")
            responses.append(content)

        except Exception as e:
            print(f"[ERROR] Failed to process input {i}: {text[:50]}\nException: {e}")
            responses.append("")

    return responses


# Hook it up to your eval
content_llms2 = {
    "mistralai/mistral-small-3.1-24b-instruct-2503:free": safe_mistral_content
}
content_results_llm = evaluate_llm("content", test_set_cc, content_llms2)
print(content_results_llm)

[DEBUG] Prompt 0: [ WP ] Four years ago , your d...
Response: In the quiet of a Thursday evening, as the last li...

[DEBUG] Prompt 1: [ WP ] In the 1700s , an etern...
Response: In the dank, fog-laden halls of the 18th century, ...

[DEBUG] Prompt 2: [ cw ] Make the reader become ...
Response: **Title: The Unspoken Symphony**

In the heart of ...

[DEBUG] Prompt 3: [ WP ] Suffering is a form of ...
Response: In the blistering heat, I stumbled through the cra...

[DEBUG] Prompt 4: [ WP ] Write about something i...
Response: In the heart of suburbia, where the houses were un...

[DEBUG] Prompt 5: [ WP ] Try to get away with mu...
Response: Title: **"Beneath the Badge"**

---

**INT. POLICE...

[DEBUG] Prompt 6: [ WP ] Iron is the only elemen...
Response: In the vast, shimmering expanse of the cosmos, Ear...

[DEBUG] Prompt 7: [ WP ] You have the power to s...
Response: Reilly was running late, as usual. She rushed down...

[DEBUG] Prompt 0: [ WP ] Write about the most un...
Response: **

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

  0%|          | 0/2 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


                                               Model      BLEU   ROUGE-L  \
0  mistralai/mistral-small-3.1-24b-instruct-2503:...  0.016447  0.130066   

   Perplexity  BERTScore_P  BERTScore_R  BERTScore_F1  
0    19.08337     0.814255     0.809367      0.811764  


In [ ]:
import evaluate
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# === 1. Load Dataset ===
from datasets import load_dataset
dataset = load_dataset("your_dataset_name", split="test")  # <- REPLACE
reference_dict = {item["prompt"].strip(): item["story"].strip() for item in dataset}  # adjust keys if needed

# === 2. Your Model Output Data ===
model_prompts = [
    "Write about something incredibly mundane , but write it as though you were writing a horror story .",
    "[ WP ] Humanity is the only `` intelligent life '' to hurt others of its kind with nothing to gain .",
    "[ WP ] You have the power to stop time but sometimes it randomly occurs , Today you find out it 's another person with the same power and you see them in the act .",
    # ... your other prompts
]

model_outputs = [
    """At 10:03 a.m., the skyline of Washington D.C. split open—not with fire or smoke, but with silence...""",
    """The desolate Earth spun silently beneath the Naevan Voyager...""",
    """The bell above the café door turned icy silent. Steam halted mid-hiss from my mug...""",
    # ... your other outputs
]

# === 3. Match references ===
matched_prompts = []
matched_outputs = []
matched_references = []

for prompt, output in zip(model_prompts, model_outputs):
    clean_prompt = prompt.strip()
    reference = reference_dict.get(clean_prompt)
    if reference:
        matched_prompts.append(clean_prompt)
        matched_outputs.append(output.strip())
        matched_references.append(reference.strip())
    else:
        print(f"[WARNING] No match found for prompt:\n{clean_prompt[:80]}...")

# === 4. Evaluate BLEU, ROUGE, BERTScore ===
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

bleu_score = bleu.compute(predictions=matched_outputs,
                          references=[[ref] for ref in matched_references])["bleu"]

rouge_score = rouge.compute(predictions=matched_outputs,
                            references=matched_references)["rougeL"]

bert = bertscore.compute(predictions=matched_outputs,
                         references=matched_references, lang="en")

bertscore_f1 = sum(bert["f1"]) / len(bert["f1"])
bertscore_p = sum(bert["precision"]) / len(bert["precision"])
bertscore_r = sum(bert["recall"]) / len(bert["recall"])

# === 5. Compute Perplexity ===
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()
if torch.cuda.is_available():
    model.cuda()

def calculate_perplexity(sentence):
    encodings = tokenizer(sentence, return_tensors='pt')
    input_ids = encodings.input_ids
    if torch.cuda.is_available():
        input_ids = input_ids.cuda()
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
    loss = outputs.loss
    return torch.exp(loss).item()

perplexities = [calculate_perplexity(text) for text in matched_outputs]
avg_perplexity = sum(perplexities) / len(perplexities)

# === 6. Print Results ===
print("\n--- Evaluation Results ---")
print(f"BLEU:         {bleu_score:.4f}")
print(f"ROUGE-L:      {rouge_score:.4f}")
print(f"BERTScore F1: {bertscore_f1:.4f}")
print(f"BERTScore P:  {bertscore_p:.4f}")
print(f"BERTScore R:  {bertscore_r:.4f}")
print(f"Perplexity:   {avg_perplexity:.2f}")



In [45]:
 #Define system prompt for LLM to rate sentiment on a 0–4 scale
system_prompt = (
    "You are a sentiment classifier. Read the review and respond with ONLY a single integer (0–4) based on the following:\n"
    "0 = Very Negative, 1 = Negative, 2 = Neutral, 3 = Positive, 4 = Very Positive.\n"
    "Reply with only the integer."
)

# Init OpenAI client via OpenRouter
ds_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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" 
)

# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Robust scoring function with fallback
def safe_mistral_sentiment(batch):
    responses = []
    for text in batch:
        try:
            response = ds_client.chat.completions.create(
                model="mistralai/mistral-small-3.1-24b-instruct-2503:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            logger.info(f"Raw response: {content}")
            print(f"INPUT: {text}\nRESPONSE: {content}\n")


            match = re.search(r"\b[0-4]\b", content)
            if match:
                label = int(match.group(0))
                responses.append(label)
            else:
                raise ValueError(f"No valid sentiment label found in response: '{content}'")

        except Exception as e:
            logger.error(f"Error processing input {len(responses)}: {text}\nException: {e}")
            responses.append(2)  # Default to Neutral
            time.sleep(1)
    return responses

# Mapping model name to function
sentiment_llms2 = {
    "mistralai/mistral-small-3.1-24b-instruct-2503:free": safe_mistral_sentiment
}
sentiment_results_llm = evaluate_llm("sentiment", test_set_sa, sentiment_llms2)
print(sentiment_results_llm)

INPUT: I've been going here a lot(pretty much a regular you could say.) It's a good place to include in your bar hopping plans. Wandos too busy? Fooseball taken at Vintage? Swing by Red Shed. It's also one of the few bars you can actually have a conversation at some  nights. One of my minor complaints is the jukebox. If there is no songs playing..it's just that--quiet. And at certain points in the night, you can hear a pin drop. Lines for the bathroom is sometimes annoying and I've felt really bad for the ladies here who have sometimes waited 30 minutes before the line moved one person :P
RESPONSE: 3

INPUT: I stalk this truck.  I've been to industrial parks where I pretend to be a tech worker standing in line, strip mall parking lots, and of course the farmer's market.  The bowls are so so absolutely divine.  The owner is super friendly and he makes each bowl by hand with an incredible amount of pride.  You gotta eat here guys!!!
RESPONSE: 4

INPUT: I LOVE Bloom Salon... all of their 

In [49]:
# Define system prompt for LLM to rate sentiment on a 0–4 scale
system_prompt = (
    "You are a sentiment classifier. Read the review and respond with ONLY a single integer (0–4) based on the following:\n"
    "0 = Very Negative, 1 = Negative, 2 = Neutral, 3 = Positive, 4 = Very Positive.\n"
    "Reply with only the integer."
)

# Init OpenAI client via OpenRouter
ds_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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" 
)

# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Robust scoring function with fallback
def safe_qwen_sentiment(batch):
    responses = []
    for text in batch:
        try:
            response = ds_client.chat.completions.create(
                model="qwen/qwen3-235b-a22b:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            logger.info(f"Raw response: {content}")
            print(f"INPUT: {text}\nRESPONSE: {content}\n")


            match = re.search(r"\b[0-4]\b", content)
            if match:
                label = int(match.group(0))
                responses.append(label)
            else:
                raise ValueError(f"No valid sentiment label found in response: '{content}'")

        except Exception as e:
            logger.error(f"Error processing input {len(responses)}: {text}\nException: {e}")
            responses.append(2)  # Default to Neutral
            time.sleep(1)
    return responses

# Mapping model name to function
sentiment_llms3 = {
    "qwen/qwen3-235b-a22b:free": safe_qwen_sentiment
}
sentiment_results_llm = evaluate_llm("sentiment", test_set_sa, sentiment_llms3)
print(sentiment_results_llm)

INPUT: Well, it's ALMOST that time of year again. My favorite celebration... aside from Thanksgiving and Halloween. It's Matsuri. \n\nYeah, the parking rates around heritage square are pretty hefty when a big event like this rolls around. But it's totally worth it. I mean, when and where else in Arizona can you stuff yourself with steamed-right-in-front-of-you nikuman, take photos with samurai, see a kimono fashion show, feel really thankful you're out of high school and no longer as susceptible to the weirdness of cosplay, buy a hapi coat, pet a bunch of shiba-inu, marvel at bonsai trees that are older than your grandmother and watch almost 60 people wail on taiko drums? \n\nOK, I'm a little biased about the drums. But hey. We get quite a crowd, so other people must like it too. ;)
RESPONSE: 4

INPUT: We were excited to eat here, it is difficult to find. They were closed at 3 p.m. on a Saturday.
RESPONSE: 1

INPUT: I stalk this truck.  I've been to industrial parks where I pretend to 